<h1><center>Modelling and Engineering of Nanoscale Materials <br> Exercises session 9: Machine Learning the PES </center></h1>
<center>massimo.bocus@ugent.be, arnout.maet@ugent.be, thomas.nicholas@ugent.be</center>

In [2]:
import numpy as np

from time import time

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.saving import load_model
tf.keras.backend.set_floatx('float64')  # keras precision
keras.utils.set_random_seed(0)  # set rng seed
tf.config.optimizer.set_jit(False)

import ase
from ase.visualize import view
from ase.io import read, write

import matplotlib.pyplot as plt

from pyiron import Project, ase_to_pyiron, pyiron_to_ase

from IPython.display import Image

<div class="alert alert-block alert-warning">
<b>Educational Objectives</b> <br> 
    
(1)  Create representative datasets to train accurate MLPs <br>
(2)  Predict ab-initio energies and forces of small molecules using simple neural networks<br>
(3)  Understand the effects of the presence/absence of physical symmetries in the descriptors of the model <br>
(4)  Evaluate the accuracy of an MLP <br>
(5)  Understand interpolation v.s. extrapolation capabilities of simple neural networks <br>
(6)  Use a self-trained MLP to optimize a water molecule and perform molecular dynamics
</div>

# Introduction

<figure>
<center><img src="https://www.dropbox.com/scl/fi/wo37pi0um6muhsd294m4e/mlp.png?rlkey=7lk0ds8w8gq3pksrrui05mlh6&st=s5bt6joy&raw=1" width="1000"></center>
<center><figcaption>Figure 1: Illustration of a general Machine Learning Potential.</figcaption></center>
</figure>

In [2]:
# Setting up the project
pr = Project('session9')

In [3]:
# A general Gaussian job function, this is easy to use for large benchmarks using for-loops
def g16_job(pr, name, jobtype, structure, lot, basis_set, charge=0, multiplicity=1, verbosity='low', settings={}, cluster='donphan', cores=1, run_time=5*60):
    '''
        ***Args***
 
        pr  the pyiron Project in which this job will be classified
 
        name (string)
            the job name, must be unique within this project
 
        jobtype (string)
            job type as defined on the Gaussian16 site: e.g. sp,opt,freq,...
 
        structure
            a pyiron structure object
 
        lot (string)
            a level of theory as defined on the Gaussian16 site
 
        basis_set (string)
            a basis set as defined on the Gaussian16 site
 
        charge (int)
            the charge of the structure
 
        multiplicity (int)
            the spin multiplicity of the structure, 2S+1

        settings (dict)
            dictionary using the following format {'key':['option1','option2',...]} such that
            e.g. the following options "int(grid=ultrafine) SCF=YQC" can be set with
            settings = {'int':['grid=ultrafine'], 'SCF':['YQC']}
 
        cluster (string)
            name of the HPC cluster on which this computation should take place
 
        cores (int)
            the number of cores which should be used (max amount of memory = cores*mem_per_cpu)
 
        run_time (int)
            wall time in seconds (max of 72*60*60)
    '''
 
    job = pr.create_job(pr.job_type.Gaussian, name, delete_existing_job=True)
    job.structure = structure
    ################################
    job.input['jobtype'] = jobtype
    job.input['lot'] = lot
    job.input['basis_set'] = basis_set
    job.input['charge'] = charge
    job.input['spin_mult'] = multiplicity
    job.input['verbosity'] = verbosity
    ################################
    if len(settings)>0:
        job.input['settings'] = settings
        
    job.server.queue = cluster
    job.server.cores = cores
    job.server.run_time = run_time
    job.run()
    # return job # you could return the job object if you want

# Exercise 1: Generating data sets

<div class="alert alert-block alert-warning">
<b>In short:</b> <br>
    1. Construct a representative dataset for the water molecule. <br>
    2. Construct an informative independent test set.
</div>

The goal of this exercise session is to build a neural network capable of predicting the ab-initio energy (and forces) of a single water molecule. A neural network takes an input vector, applies a series of linear and nonlinear transformations, and produces one or more output values. These internal transformations are determined by the model’s weights and biases (collectively called the parameters), which control how the numerical values (features) in one layer are mapped to the next.

During training, the network’s parameters are iteratively adjusted so that the model’s outputs match the desired results as closely as possible. Every neural network, from machine-learning potentials (MLPs) to large language models (LLMs) such as ChatGPT, relies on this optimization process to achieve good predictive performance.

In our case, we aim to construct a neural network that can take the geometry of a water molecule and predict its energy (and forces) at a chosen level of theory. To do this, we need a dataset of water configurations labeled with their “correct” ab-initio energies. Neural networks excel at interpolating between data points but perform poorly at extrapolation. This means our training set must adequately cover the region of phase space we intend to explore; if the model encounters configurations outside this region, its predictions will be highly unreliable. Additionally, we will prepare an independent test set to evaluate the accuracy of the trained model.

<div class="alert alert-block alert-success">
<b>Answer the following questions using the code below:</b> <br>   
<ol type='1'>
<li> Create a training set for water at the MP2/6-31g level of theory containing 50 structures. What do you think constitutes a good training set? Visualise your data set by plotting the distribution of the relevant internal coordinates.</li>
<br>
<li> Create an independent test set of 50 structures by scanning one of the O-H distances. Make sure that the range of distances goes beyond those present in the training set, such that you can see the extrapolated/interpolated points. What can we learn from this test set?</li>
<br>
<li> Why is evaluating the training set not a reliable way to quantify the accuracy of a neural network?</li>
</ol>
</div>

### Question 1

Load the optimized water structure from session 1 and create a dataset of $50$ structures by manipulating the optimized structure.

**HINT:** Try to make the dataset sufficiently diverse, the model can not learn anything new from duplicate structures. However, do not make it too diverse as this will make training more difficult.

<div class="alert alert-block alert-info">
<b>Data structure</b>

It will be convenient to store the structures and their corresponding energy as a list of `ase.Atoms`-objects:

```python
configs = []  # list to store Atoms-objects

job = ex1.load('job')
structure = job.structure
energy = job.output.energy_tot[-1]  # extract energy from job

atoms = pyiron_to_ase(structure)
atoms.info['energy'] = energy  # add energy to Atoms-object

configs.append(atoms))  # add Atoms-object to list
```

The energy of `ase.Atoms`-objects can be called as:

```python
energy = config.get_potential_energy()
```

Lists of `ase.Atoms`-objects can then also be saved to or loaded from disk using the `write`- and `read`-functions:

```python
write('path.xyz', configs)
configs = read('path.xyz', index=':')
```

</div>

In [4]:
ex1 = pr.create_group('ex1')
water_opt = ...  # optimzed water molecule from session 1

In [5]:
# add code here to manipulate water molecule

In [6]:
# add code here to evaluate the structures

In [8]:
# add code here to save structures as .xyz file

### Question 2

In [7]:
# add code here to scan an internal coordinate

In [ ]:
# add code here to evaluate structures

In [ ]:
# add code here to save structures as .xyz file

### Question 3

# Exercise 2a: Model without explicit symmetries.

<div class="alert alert-block alert-warning">
<b>In short:</b> <br>
    1. Construct a neural network without explicit physical symmetries  <br>
    2. Evaluate the accuracy of the network 
</div>

Now that we have our labeled training set, we can start building our first neural network potential! As an initial attempt, let us create a model which takes in atomic positions and and returns (relative) energy. To do this, we will use the Keras API which has a plethora of helper functions built around the three main deep learning frameworks in Python: PyTorch, TensorFlow and JAX. Keras will allow us to efficiently create and train neural networks without having to fully understand the underlying code of these backends. As this is not a deep learning course, most of the code implementation will be provided for you. However, feel free to enter the rabbit hole of [Keras](https://keras.io/) and [TensorFlow](https://www.tensorflow.org/) to quench your unavoidable curiosity!

<div class="alert alert-block alert-success">
<b>Answer the following questions using the code below:</b> <br>   
<ol type='1'>
<li> Load in your training set and select 10 structures to be used for the validation set. How can overfitting be detected by tracking the validation loss?</li>
<br>
<li> Construct the layout of the model and train the weights on your training set, once with the absolute energies and once with relative energies (so subtracting the average energy for example). Which approach leads to the most efficient/stable training and can you think of why this is?</li>
<br>
<li> Continue with the best approach from the previous question. Compute the root mean square error on the training, validation and test set. For these data sets, make a parity plot and plot the reference and predicted energies together as a function of a relevant internal coordinate. Do you see a big difference in the model accuracy on structures inside as opposed to outside of the training set?</li>
<br>
<li> Do you get the same energy after translating or rotating the water molecule? If not, why is this?</li>
</ol>
</div>

### Question 1

Let's start by loading in the training set and selecting randomly 20% of the structures for the validation set. The model is not trained on the validation set, such that it can be used to gauge the ability of the model to generalise to unseen structures. 

<div class="alert alert-block alert-info">
<b>Visualising data</b>

Data sets of molecular structures can be visualised as a function of its state, which can be characterized by some of its internal coordinates. In our case, it is obvious that we can plot the energy as a function of the bond distances and angles. For example, for O-H1 distance we might write:

```python
distances = []
energies = []
for config in data_set:
    distance = config.get_distance(0, 1)
    energy = config.get_potential_energy()
    distances.append(distance)
    energies.append(energy)

plt.plot(distances, energy, 'o'))
```

</div>

In [10]:
# add code here

To get a sense of the phase space coverage, it can be helpful to visualize your training sets by plotting histograms or the energy as a function of some internal coordinates. Where are your validation structures in these visualizations?

In [9]:
# add code here

### Question 2

Transform these `ase.Atoms`-objects into training input (here: just atomic positions) as a array/tensor with shape $(B, N, 3)$, where $B (=50)$ is the amount of structures in the training set, N is the amount of atoms in each structure ($=3$) and $3$ represents the three spatial coordinates. Also extract the output labels (energies) as a tensor with shape (B, ), i.e. one energy per training structure.

<div class="alert alert-block alert-info">
<b> Input tensors </b>

To transform our raw input into tensors that the model can use, we can use `numpy` to get the dimensions right. For example, if we have ten structure each with three atoms, the input array should have shape (10, 3, 3) and the output array shape (10, 1):

```python
x = np.zeros((10, 3, 3))
y = np.zeros((10, 1))
for i, structure in data_set:
    x[i, :, :] = structure.get_positions()
    y[i] = structure.get_potential_energy()
```

Afterwards, TensorFlow prefers that you convert these `numpy`-arrays into `tf.Tensor`-objects:

```python
x = tf.constant(x)  # convert numpy-array into tf.Tensor
y = tf.constant(y)
```
</div>

In [23]:
x_train = np.ones(...)   # input: (B, N, 3)
y_train = np.ones(...)   # output: (B, 1)
x_valid = np.ones(...)
y_valid = np.ones(...)

Let us now create a sequential neural network using the Keras functionalities, consisting of four layers: an input layer with $9$ nodes corresponding to the spatial coordinates, two dense hidden layers with $10$ nodes and an output layer with a single node to read out the energy. Figure 2 shows schematically how such a network looks; the spatial coordinates of the atoms are flattened into a vector of length $9$ and are passed through the model, which returns the predicted energy as output. Each string connecting the layers represents a single weight. 

<figure>
<center><img src="https://www.dropbox.com/scl/fi/oa1uc8lqxmqsoy036lzwo/neural_network1.png?rlkey=qn406uz8o2oitbiis2lovgs3e&st=01myle8k&raw=1" width="1000"></center>
<center><figcaption>Figure 2: Schematic view of the neural network constructed in this exercise.</figcaption></center>
</figure>

In [24]:
input_shape = (3, 3)    # (N, 3) where N=3 for H2O
hidden_initializer = keras.initializers.HeNormal(seed=0)    # initialization of the weights in the hidden layers, fixed seed for reproducibility
output_initializer = keras.initializers.RandomUniform(seed=0, minval=-1e-3, maxval=1e-3)    # initialization of the weights in the output layer, fixed seed for reproducibility
model_absolute = keras.Sequential()     # initialize model of sequential layers
norm_layer = keras.layers.Normalization(axis=-1)    # normalization layer to center inputs around zero
norm_layer.adapt(x_train)
model_absolute.add(keras.Input(shape=input_shape))  # input layer
model_absolute.add(norm_layer)
model_absolute.add(keras.layers.Flatten())   # flatten input into one-dimensional tensor of length 9
model_absolute.add(keras.layers.Dense(10, activation='swish', kernel_initializer=hidden_initializer))
model_absolute.add(keras.layers.Dense(10, activation='swish', kernel_initializer=hidden_initializer))
model_absolute.add(keras.layers.Dense(1, activation='linear', kernel_initializer=output_initializer))
print(model_absolute.summary())

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ normalization_4 (Normalization) │ (None, 3, 3)           │             7 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_4 (Flatten)             │ (None, 9)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 10)             │           100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 10)             │           110 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 1)              │            11 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 228 (1.78 KB)

 Trainable params: 221 (1.73 KB)

 Non-trainable params: 7 (56.00 B)

None


For further convencience, create a function which takes in the training points, the input shape and network layout and creates a model, similar to the function you created for submitting Gaussian jobs.

In [11]:
def create_nn(x_train, input_shape=(3, 3), neurons_list = [10, 10]):
    # fix me
    return model

<div class="alert alert-block alert-info">
<b>Network size</b>

The size of a neural network is usually reported as the number of (un)trainable parameters, this determines the computational cost of a single forward pass. As we are working on a very simple system here, models with under 300 parameters are already very accurate. As a reference, modern day universal MLPs can have around 30 million parameters ([link](https://matbench-discovery.materialsproject.org/)), while big LLMs already have over 1 trillion parameters ([link](https://www.labellerr.com/blog/comparing-language-models-through-parameters-vs-real-life-experiments/))!
</div>

<div class="alert alert-block alert-info">
<b>Units</b>

Keep in mind that the model has no understanding of units and it will learn the PES in the units which are used in the training set, therefore makes sure to be consistent. If you're training set uses Angstrom you must always supply coordinates in Angstrom. Same thing for the output; if the energy in the training set was in eV, the model will always output in this unit. Therefore, keep track of any offsets or normalizations you apply!
</div>

Now that we have constructed the layout of our neural network, we can start training. The parameters are optimized by minimizing a so-called loss function, in this case this is the mean squared error between the reference and predicted energies. In principle, you should continue training until the (validation) loss has converged, but this might take quite long depending on the model and training set. To check training efficiency, it can sometimes be helpful to train for a fixed amount of epochs.

**NOTE:** re-executing the cell below will **continue** training, not **restart** it. If you wish to reset the model, you should redefine the model layout (above cell). 

<div class="alert alert-block alert-info">
<b>Learning rate</b>

The learning rate determines how strongly the parameters are changed each epoch (training step). In the early stages of training this should be high, so that the model quickly learns the overall trends of the dataset. If you want to train as optimally as possible, you should decrease the learning rate each time the (validation) loss converges.
</div>

In [ ]:
from time import time
# Initial 'rough' training with higher learning rate
model_absolute.compile(
    loss=keras.losses.MeanSquaredError(reduction="sum_over_batch_size", name="mse"),
    optimizer=keras.optimizers.Adam(learning_rate=1e-2),
)
epochs = 2000
t0 = time()
model_absolute.fit(x_train, y_train, batch_size=5, epochs=epochs, validation_data=(x_valid, y_valid))
t1 = time()
print(f"Training time (s) for {epochs}:", t1 - t0)

As before, create a function to train a model.

In [26]:
def train_model(model, x_train, y_train, x_valid, y_valid, learning_rate=1e-3, epochs=2000):
    # fix me
    return model

The loss is very high! Construct the output tensors once more but with the average energy substracted. 

In [ ]:
x_train = np.ones(...)   # input: (B, N, 3)
y_train = np.ones(...)   # output: (B, 1)
x_valid = np.ones(...)
y_valid = np.ones(...)
average energy = ...

Create a new model and train again.

In [28]:
model_relative = create_nn(x_train)
print(model_relative.summary())

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ normalization_5 (Normalization) │ (None, 3, 3)           │             7 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_5 (Flatten)             │ (None, 9)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 10)             │           100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 10)             │           110 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 1)              │            11 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 228 (1.78 KB)

 Trainable params: 221 (1.73 KB)

 Non-trainable params: 7 (56.00 B)

None


In [ ]:
model = train_model(model_relative, x_train, y_train, x_valid, y_valid)

**CONGRATULATIONS**, you have just built and trained your very own neural nework potential, we are very proud of you! Now save it, before it disappears forever...

<div class="alert alert-block alert-info">
<b>Saving and loading models</b> <br>

After training the model is stored in the temporary memory of the notebook as a Python object, thus if anything happens to the Python kernel, the model will be lost and will need to be retrained. Therefore, it might be wise to save your models, this can also be helpful when directly comparing different models.

```python
model_path = "path/to/model.keras"
model.save(model_path)
```
Make sure the add the .keras extension to the path.
Models can then be loaded like:

```python
from tensorflow.keras.saving import load_model
model = load_model(model_path)
```
</div>

In [30]:
# add code to save the model

### Question 3

To get a first sense of the accuracy of the model, we can evaluate the training set and create a parity plot. A parity plot visualizes the error by plotting the points at $(E_{dft}, E_{mlp})$, a theoretical perfect model would have each point on the diagonal such that $E_{dft} = E_{mlp}$. It could also be helpful to plot the reference and predicted energies as a function of their index or some relevant internal coordinates. Are specific energies predicted much more poorly than the rest and what structures do they correspond with? Is this a surprise?

**HINT:** create a function that can evaluate the error of a model on a dataset and visualise the reference and predicted energies.

<div class="alert alert-block alert-info">
<b>Error metrics</b>

The accuracy of an MLP is conventionally reported as the root mean squared error between the reference and predicted energies:

$ E_\text{rmse} = \sqrt{\frac{1}{B}\sum_{i = 1}^B (E_\text{ref, i} - E_\text{mlp, i})^2} $, where B is the amount of structures in the evaluated dataset.

It would only make sense to compare the $E_\text{rmse}$ for systems with equal size, therefore it is usually normalized to the amount of atoms. If the number of atoms is fixed in the evaluated dataset:

$ E_\text{rmse, norm} =\frac{1}{N} E_\text{rmse} $ in meV/atom, where N is the amount of atoms in the structures.

Present day (non-universal) MLPs are considered 'very accurate' when their RMSE is below 1 meV/atom.

</div>

**HINT:** to evaluate a dataset using your model, transform the input into a tensor as you did with the training set and call the `model.predict()`-function.

```python
predicted_energies = model.predict(x_test) + reference_energy
```

<div class="alert alert-block alert-info">
<b>Shape matters</b>

Input and intermediate values are internally represented using tensors which can often have complex, high-dimensional shapes. Keras will always expect the first index of the input tensor to be a batch index, which corresponds to the index of the specific sample (here molecular structure) in the total data set. So, if we want to evaluate $B$ structures and each structure has $N$ atoms with $3$ spatial coordinates, the model will expect the input tensor to have shape $(B, N, 3)$. The expected output will be a tensor of shape $(B, 1)$, which is not the same as a one-dimensional tensor of length $B$! 

**NOTE:** this means that if you want to evaluate a single structure, the tensor should have a batch length of 1, i.e. $(1, N, 3)$ instead of $(N, 3)$.

</div>

In [12]:
# add code here to see how well the reference and predicted energies match

Now evaluate on the test set!

In [13]:
# add code here to see how well the reference and predicted energies match

### Question 4

In [14]:
# add code here to see what happens to the energy when we translate/rotate the structure


# Exercise 2b: Incorporating geometric symmetries.

<div class="alert alert-block alert-warning">
<b>In short:</b> <br>
    1. Construct a neural network that respects rototranslational invariance  <br>
    2. Evaluate the accuracy of the network and observe the improvements
</div>

To be able to pass a molecular structure through the neural network, we need to transform that structure into a numerical input tensor/vector, called a descriptor. This process of transforming the atomic coordinates (and corresponding species) into rich descriptive feature vectors is called embedding and is an entire research topic in itself. A good embedding should uniquely, richly and efficiently decribe the system (or the chemical environment of an atom) and respect the fundamental symmetries. Sophisticated MLP architectures can have very complex embedding schemes, for example MACE uses message passing neural networks to learn the embedding. The most basic and intuitive descriptor was handled in the previous exercise, where we simply use the spatial coordinates of each atom as input. As you saw, this does not give a transferable model, because we basically incorporate no physical symmetries into the descriptor of the system. We can quite easily improve on this by using descriptors which have rototranslational symmetry. There are several different possibilities, can you think of a descriptor for the molecular structure which respects the global rotation and rotation invariance? Remember that the number of independent degrees of freedom is $3N-6$ ($=3$), therefore you should be able to completely describe the state of the water molecule using $3$ parameters.

<div class="alert alert-block alert-success">
<b>Answer the following questions using the code below:</b> <br>   
<ol type='1'>
<li> Can you think of a descriptor for a water molecule that respects the rototranslational symmetry? Adapt the input and the network layout to handle this, remember that the shape of the input will differ.</li>
<br>
<li> Train a model using this decriptor. What happens to the training efficieny, accuracy and transferability?</li>
<br>
<li> Are we missing any additional fundamental symmetries? What happens when you swap the two hydrogen atoms? Do you notice any other flaws with this model and the previous one?</li>
</ol>
</div>

### Question 1

**HINT:** If you use a descriptor with three parameters, the input shape of the model will change from $(3, 3)$ to $(3, )$.

In [60]:
# add code here to convert the descriptor of each structure into a tf.Tensors

### Question 2

### Question 3

In [15]:
# add code here to see what happens to the energy when we swap the two hydrogen atoms

# Exercise 3: Computational cost versus accuracy

<div class="alert alert-block alert-success">
<b>Answer the following questions using the code below:</b> <br>   
<ol type='1'>
<li> What is the effect of the network layout/size on the training and model performance? Do you notice underfitting/overfitting? Why should you not make your models too large?</li>
<br>
<li> What is the effect of the training set size on the training and model performance?</li>
</ol>
</div>

As you have seen the computational cost of a forward pass is determined by the size of your model (# of parameters), because this directly relates to the numbers of tensor operations that the model performs. Importantly, the number of tunable weights in the neural network will also determine the expressivity and transferability of your model. More weights means more freedom of the model to fit the training set accurately. 

Beware of too much freedom, however!

### Question 1

In [16]:
# SMALL MODEL

In [17]:
# LARGE MODEL

### Question 2

# Exercise 4: Incorporating permutation invariance and forces

<div class="alert alert-block alert-warning">
<b>In short:</b> <br>
    1. Construct a neural network that respects permutation invariance  <br>
    2. Incorporate the ability to calculate forces <br>
    3. Perform optimizations and MD simulations using your own neural network
</div>

### Permutation invariance

Another fundamental physical invariance is permutation invariance, i.e. swapping two identical atoms (2 H atoms) will not change the total energy of the system. Up until now, our descriptors have depended on the order of the atoms in the input. Permuation invariance can be explicitly incorporated by, instead of predicting one total energy, we predict fictional 'per-atom energies' for each atom and sum them all together to obtain the total energy. It is clear that the explicit permutation invariance is inherited from the commutativity of the sum. These per-atom energies are unphysical, but their individual values are irrelevant, the only thing that matters is how they sum together.

**NOTE:** the per-atom energies mentioned here are entirely unrelated to the energies of the isolated atoms. 

To get atomic energies, we need an input feature (descriptor) for each atom individually. In general, this atomic descriptor should contain all of the information necessary to describe the (local) chemical environment of that specific atom. We can then pass these atomic features one by one through a single shared model and sum all of their energies together to obtain the total energy of the structure. This is schematically shown in Figure 4. In a permutation invariant model it is important that the atomic feature also contains information on the type of atom, otherwise the model will think that each atom is identical. For a specific atom, we can define the atomic descriptor as the distances between that atom and the other two atoms, and extend it with a one-hot encoding vector to account for the atomic species $(O = [0, 1], H = [1, 0])$.

<figure>
<center><img src="https://www.dropbox.com/scl/fi/xqsdlzzgkeg806lldu7wh/energy_readouts.png?rlkey=0b1lo9zvn0vn72mz46oi4cdly&st=b46bd8zo&raw=1" width="700"></center>
<center><figcaption>Figure 3: Schematic representation a permutation invariant MLP, atomic energies are calculated through the same model and summed to obtain the total energy.</figcaption></center>
</figure>

### Forces

Neural networks are usually trained using a technique called **backpropagation**. In this algorithm, the gradient of the loss function w.r.t. the model weights is efficiently calculated using the chain rule of calculus. The prediction error is propagated backwards through the network to determine how to tweak the weights and biases to minimize the error. In practice, this is enabled by the built-in **automatic differentiation** implemented in TensorFlow. All tensor operations are (at least once) continuously differentiable, such that the entire network is actually a non-linear analytic function that can be differentiated, if all intermediate information is stored after a forward pass. Fortunately for us, this automatic differentiation can also be used to calculate the forces acting on the atoms, as this is simply the **negative** gradient of the energy w.r.t. the positions. The forces acting on atom $i$ are given by

$ \vec{F_i} = -(\frac{\partial E}{\partial x_i}, \frac{\partial E}{\partial y_i}, \frac{\partial E}{\partial z_i}) $.

Therefore, we automatically have forces, as long as each operation from atomic positions to total energy is differentiable, which is the case for (most of) the TensorFlow tensor operations.

<div class="alert alert-block alert-info">
<b>Activation functions</b>

Activation functions are non-linear functions that are applied to the weights of each layer. It is precisely this non-linearity that allows the model to learn complex relations between input and output. Importantly, not all activation functions are actually continuously differentiable, for example the well known ReLU function. For this reason, we have mainly used the swish activation function which is basically a smoother version of ReLU and yields better gradients for training and forces.
</div>

<div class="alert alert-block alert-success">
<b>Answer the following questions using the code below:</b> <br>   
<ol type='1'>
<li> Train a model with this new architecture and use it to optimize some structures from the training set. What are the optimized bonds, angles?</li>
<br>
<li> Perform some NVT simulations at room temperature. Visualise the trajectory and plot the bond lengths, angles, does this look realistic to you?</li>
<br>
<li> Can you get a measure of how much more efficient this neural network potential is in comparison to DFT calculations on a water molecule?</li>
</div>

In [43]:
from tensorflow.keras import saving

# normalization of the pair-wise distances
def get_mean_std(input):
    descriptor_layer = StablePairwiseDistancesWithSpecies()
    desc = descriptor_layer(input)
    distances = desc[:, :, :2]
    d_mean = tf.reduce_mean(distances, axis=[0,1])
    d_std = tf.math.reduce_std(distances, axis=[0,1])

    return d_mean, d_std

@saving.register_keras_serializable()
class CustomNormalization(tf.keras.layers.Layer):
    def __init__(self, mean, std, **kwargs):
        super(CustomNormalization, self).__init__(**kwargs)
        self.mean = tf.constant(mean, dtype=tf.float64)
        self.std = tf.constant(std, dtype=tf.float64)
        
    def call(self, inp):
        distances = inp[:, :2]
        onehot = inp[:, 2:]
        d_norm = (distances - self.mean) / self.std
        return tf.concat([d_norm, onehot], axis=-1)

    def get_config(self):
        config = super().get_config()
        # You must convert tensors to lists for serialization
        config.update({
            "mean": self.mean.numpy().tolist(),
            "std": self.std.numpy().tolist(),
        })
        return config

    @classmethod
    def from_config(cls, config):
        # Keras will pass mean/std back as lists
        return cls(**config)

@saving.register_keras_serializable()
class StablePairwiseDistancesWithSpecies(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super(StablePairwiseDistancesWithSpecies, self).__init__(**kwargs)

    def call(self, coords):
        """
        coords: (B, N, 3)
        returns: (B, N, N-1)
        """
        B = tf.shape(coords)[0]
        N = tf.shape(coords)[1]

        # Compute pairwise vectors: (B, N, N, 3)
        diff = coords[:, :, None, :-1] - coords[:, None, :, :-1]

        # Pairwise distances: (B, N, N)
        dist_mat = tf.norm(diff, axis=-1)

        # Create a diagonal mask: (N, N)
        diag_mask = ~tf.eye(N, dtype=tf.bool)

        # Broadcast mask to (B, N, N)
        diag_mask = tf.broadcast_to(diag_mask, tf.shape(dist_mat))

        # Mask out the diagonal, flatten to (B, N*(N-1))
        flat = tf.boolean_mask(dist_mat, diag_mask)

        # Reshape to (B, N, N-1)
        res = tf.reshape(flat, (B, N, N - 1))

        species = coords[:, :, -1]
        # one hot encode species
        species = keras.layers.CategoryEncoding(num_tokens=2, output_mode='one_hot')(tf.cast(species, tf.int32))
        # add columns in dists for species
        desc = tf.concat([res, species[:, :, 0:1], species[:, :, 1:2]], axis=-1)

        return desc

@saving.register_keras_serializable()
class SumLayer(tf.keras.layers.Layer):
    def __init__(self, axis=1, **kwargs):
        super().__init__(**kwargs)
        self.axis = axis

    def call(self, inputs):
        return tf.reduce_sum(inputs, axis=self.axis)

    def get_config(self):
        config = super().get_config()
        config.update({"axis": self.axis})
        return config

### Question 1

In this model, we want to explicitly encode the atom species into the descriptor, such that the model does not have to learn this itself. The input tensor of the model must now be structured in a way that includes the atomic number of each atom to be evaluated. The model input will be the position vector of the three atoms, extended with the atomic number of that atom. However, the atomic number can be high and neural networks do not like this. We will therefore use $0$ for the oxygen atoms and $1$ for the hydrogen atoms. Such that for a single structure, the input tensor $\textbf{I}$ has shape (3, 4) and should look like:

\begin{equation}
\textbf{I} = 
\begin{bmatrix}
x_1 & y_1 & z_1 & \alpha_1 \\
x_2 & y_2 & z_2 & \alpha_2 \\
x_3 & y_3 & z_3 & \alpha_3 \\
\end{bmatrix},
\end{equation} where $\alpha = {0, 1}$. 

We have prepared some custom layers to transform this input tensor into a good descriptor for the model.

**HINT:** the atomic numbers can be extracted using `Atoms.get_atomic_numbers()` or `Atom.number`:

```python
atomic_number1 = water.get_atomic_numbers()[0]
alpha1 = numbers2type[atomic_number1]
```

or

```python
atomic_number1 = water[0].number
alpha1 = numbers2type[atomic_number1]
```

In [ ]:
x_train = np.ones((len(train_set), len(train_set[0]), 4))   # input: (B, N, 3 + 1 for atom number)
y_train = np.ones((len(train_set), 1))                          # output: (B, 1)
x_valid = np.ones((len(valid_set), len(valid_set[0]), 4))
y_valid = np.ones((len(valid_set), 1))
numbers2type = {8: 0, 1: 1}  # O:0, H:1

# fix me


In [100]:
mean, std = get_mean_std(x_train)
print("Distance mean:", mean.numpy())
print("Distance std:", std.numpy())
input_shape = (3, 4)
model3 = keras.Sequential()

model3.add(keras.layers.Input(shape=input_shape))
model3.add(StablePairwiseDistancesWithSpecies())
hidden_layers = keras.Sequential([
    keras.layers.Input(shape=(4,)),
    CustomNormalization(mean, std),
    keras.layers.Dense(10, activation='swish', kernel_initializer=hidden_initializer),
    keras.layers.Dense(10, activation='swish', kernel_initializer=hidden_initializer),
    keras.layers.Dense(1, activation='linear', kernel_initializer=output_initializer),
])

model3.add(keras.layers.TimeDistributed(hidden_layers))
sum_layer = SumLayer()
model3.add(sum_layer)
print(model3.summary())

Distance mean: [0.98544451 1.36322112]
Distance std: [0.15040774 0.33188506]


Model: "sequential_18"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ stable_pairwise_distances_with… │ (None, 3, 4)           │             0 │
│ (StablePairwiseDistancesWithSp… │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_2              │ (None, 3, 1)           │           171 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sum_layer_2 (SumLayer)          │ (None, 1)              │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 171 (1.34 KB)

 Trainable params: 171 (1.34 KB)

 Non-trainable params: 0 (0.00 B)

None


In [18]:
# add code here to train model

In [19]:
# add code here to evaluate the accuracy of the model

Our entire neural network, from input (positions) to output (energy), is fully automatically differentiable. Therefore, we can now calculate the forces by computing the gradient of the energy w.r.t. the atomic coordinates. In TensorFlow, this is done like this:

```python
input_tensor = tf.Variable(input_tensor, dtype=tf.float64)  # Shape: (1, 3, 4). This tells TensorFlow to store intermediate results for the gradient computation
with tf.GradientTape() as tape:
    energy_prediction = model(input_tensor)
gradients = -tape.gradient(energy_prediction, input_tensor) # (1, 3, 4) shaped tensor containing gradients.
```

By ignoring the last column, we obtain the three force components acting on each atom:

```python
forces = gradients[:, :, :-1]
```

### ASE Calculator
By creating a minimal ASE Calculator for our neural network potential, we can couple our energy and force calculations to ASE. This allows us to interface with the optimization and molecular dynamics functions from ASE.

In [104]:
from ase.calculators.calculator import Calculator, all_changes
from ase import Atoms
from ase.parallel import broadcast, world
class NNPCalculator(Calculator):
    """
    ASE Calculator wrapper to calculate energies and forces using a simple neural network potential.
    """
    implemented_properties = ['energy', 'forces']

    def __init__(self, model, atoms: Atoms = None):
        self.istep = 0
        Calculator.__init__(self, atoms=atoms)
        self.model = model

    def _get_name(self):
        return 'NNP'

    def calculate(self, atoms: Atoms = None, properties: list[str] = ['energy', 'forces'], system_changes=all_changes):
        Calculator.calculate(self, atoms, properties, system_changes)

        positions = atoms.get_positions()
        comp = self.compute_energy_and_forces(positions, self.istep)
        energy, forces = comp
        self.istep += 1
        self.results['energy'] = energy
        self.results['forces'] = forces[:, :3]

    def compute_energy_and_forces(self, pos, istep):
        input_tensor = np.zeros((1, pos.shape[1], 4), dtype=np.float64)
        input_tensor[0, :, :-1] = pos
        numbers2type = {8: 0, 1: 1}  # O:0, H:1
        input_tensor[0, :, -1] = np.array([numbers2type[atom.number] for atom in self.atoms])
        pos = tf.Variable(input_tensor, dtype=tf.float64)
        if world.rank == 0:
            with tf.GradientTape() as tape:
                energy_pred = self.model(pos) 
            forces_pred = -tape.gradient(energy_pred, pos)
            ener_force = (energy_pred.numpy().squeeze(), forces_pred.numpy().squeeze())
        else:
           ener_force = None
        energy, forces = broadcast(ener_force)
        return energy, forces

    def __enter__(self):
        return self

<div class="alert alert-block alert-info">
<b>ASE calculators</b>

An `ase.Atoms`-object can have a `calc`-attribute which computes (at least) the energy and forces used for optimisations, MD, etc. In our case, we have made a custom ASE calculator (`NNPCalculator`) that can compute the energy and forces through our own neural network. Calculators can be attached to `ase.Atoms`-objects as follows:

```python
model = load_model('path/to/model')
calculator = NNPCalculator(model)
config = read('path/to/some/config')   # Atoms object
config.calc = calculator
```

Then `config` can be used for optimisations or MD.
</div> 

Use the `BFGS`-optimizer from ASE to optimize structures from the test set, for more information check the [manual](https://ase-lib.org/ase/optimize.html#ase.optimize.BFGS).

In [21]:
from ase.optimize import BFGS
def optimize(config, calculator, fmax=1e-4):
    # fix me
    return

### Question 2

Use the MD engine in ASE to perform a short 500 fs MD simulation in the NVT ensemble at 300 K with a water molecule, for more information see the [manual](https://ase-lib.org/ase/md.html#module-ase.md.langevin)

In [22]:
from ase.md.langevin import Langevin
def nvt(config, calculator, steps=1000, temperature=300, n_print=10):
    # fix me
    return

### Question 3